# SysML v2 -> ArangoDB -> answers in English

Three SysML models become a graph, and then you ask it questions. Needs a local
ArangoDB and an OpenAI key:

```bash
docker run -d --name christian-webb-drone-arango -p 8529:8529 \
  -e ARANGO_ROOT_PASSWORD=testpass arangodb:3.12.9.4 \
  arangod --experimental-vector-index=true
export CHAT_API_KEY=sk-...
```

## 1. Build it

`parse` reads the `.sysml` sources, `project` writes them into the importer's
collections, `enrich` adds communities and embeddings. Embeddings are cached, so a
second run costs nothing.

In [1]:
import contextlib, io, logging

from sysml import config, nl
from sysml.pipeline import enrich, parse, project

logging.disable(logging.INFO)  # the services narrate every step; keep just the answers

with contextlib.redirect_stdout(io.StringIO()):
    parse.main()
    project.main()
    enrich.main()

db = config.db()
for name in config.ALL_COLLECTIONS:
    print(f"{db.collection(name).count():>6}  {name}")

    30  sysml_Documents
   200  sysml_Chunks
  2359  sysml_Entities
    44  sysml_Communities
  9764  sysml_Relations


Every edge carries two fields, holding two separate vocabularies.

`type` is the importer's, and it is closed -- five constants that say how the corpus
is wired together. They are the same five in any GraphRAG corpus, whatever it is
about. `RELATED_TO` is the single bucket for "these two things are related", because
the importer cannot know what related means in someone else's domain.

In [2]:
STRUCTURE = f'''
FOR r IN {config.RELATIONS}
  COLLECT kind = r.type WITH COUNT INTO n
  SORT n DESC RETURN {{kind, n}}'''

for row in db.aql.execute(STRUCTURE):
    print(f"{row['n']:>6}  {row['kind']}")

  5300  RELATED_TO
  2284  MENTIONED_IN
  1944  IN_COMMUNITY
   200  PART_OF
    36  SUB_COMMUNITY_OF


`relationship_type` is where the domain's own word goes, and on this graph that word
is SysML's. It is set on `RELATED_TO` edges and nowhere else, so grouping by it counts
the authored relations and skips the structural wiring.

This is the whole reason for parsing rather than extracting. `satisfy R by S` is a
statement the source makes, so `satisfies` is a fact rather than a model's reading of
what some prose seemed to imply.

In [3]:
RELATIONS = f'''
FOR r IN {config.RELATIONS}
  FILTER r.type == "RELATED_TO"
  COLLECT kind = r.relationship_type WITH COUNT INTO n
  SORT n DESC LIMIT 8
  RETURN {{kind, n}}'''

for row in db.aql.execute(RELATIONS):
    print(f"{row['n']:>6}  {row['kind']}")

  2251  owns
   989  typedBy
   777  specializes
   368  refines
   263  satisfies
   184  redefines
   165  imports
   105  performs


## 2. Ask it in AQL

AQLizer writes a query, runs it, and explains the rows. Good at counting, gaps and
anything you would otherwise write AQL for. The query it used is always shown, because
a query that is subtly wrong returns no rows, and a fluent sentence about no rows
reads exactly like a correct answer about something genuinely absent.

In [4]:
nl.instance().ask("Which requirements in the drone-base model does nothing satisfy?").show()

Q  Which requirements in the drone-base model does nothing satisfy?

AQL
   WITH sysml_Entities
   FOR e IN sysml_Entities
     FILTER e.entity_type IN ["RequirementUsage", "RequirementDefinition"]
     FILTER e.model == "drone-base"
     LET satisfiers = LENGTH(
       FOR r IN sysml_Relations
         FILTER r._to == e._id AND r.relationship_type == "satisfies"
         LIMIT 1 RETURN 1)
     FILTER satisfiers == 0
     RETURN {entity_name: e.entity_name,
             at: CONCAT(e.source_file, ":", e.source_line)}

rows (3, first 3)
   {"entity_name": "Drone_SystemRequirements::totalMass", "at": "Drone_BaseArchitecture.sysml:30"}
   {"entity_name": "Drone_SystemRequirements::battery", "at": "Drone_BaseArchitecture.sysml:36"}
   {"entity_name": "Drone_SystemRequirements::maxCapacity", "at": "Drone_BaseArchitecture.sysml:39"}

A  The following requirements in the drone-base model do not have any satisfying components:

1. Drone_SystemRequirements::totalMass (Drone_BaseArchitecture.sysm

That one is a gap in the model. This one is arithmetic over a traversal -- the stages
are three levels down from the vehicle, and nobody wrote the total anywhere.

In [5]:
nl.instance().ask("What is the total dry mass of the Saturn V, summed from its stages?").show()

Q  What is the total dry mass of the Saturn V, summed from its stages?

AQL
   WITH sysml_Entities, sysml_Relations
   FOR e IN sysml_Entities
     FILTER e.name == "SaturnV"
     LET parts = (
       FOR child, edge IN 1..6 OUTBOUND e sysml_Relations
         FILTER edge.relationship_type IN ["owns", "typedBy"]
         FILTER child.attributes.dryMass.value != null
         RETURN DISTINCT {mass: child.attributes.dryMass.value}
     )
     RETURN {total_dry_mass: SUM(parts[*].mass)}

rows (1, first 1)
   {"total_dry_mass": 188650}

A  The total dry mass of the Saturn V, summed from its stages, is 188,650 units.



And this one is a shape question about the model itself.

In [6]:
nl.instance().ask("Which entity types are the most common in the Apollo model? Top 6 with counts.").show()

Q  Which entity types are the most common in the Apollo model? Top 6 with counts.

AQL
   WITH sysml_Entities
   FOR e IN sysml_Entities
     FILTER e.model == "apollo-11"
     COLLECT entityType = e.entity_type WITH COUNT INTO count
     SORT count DESC
     LIMIT 6
     RETURN {entity_type: entityType, count: count}

rows (6, first 6)
   {"entity_type": "AttributeUsage", "count": 423}
   {"entity_type": "RequirementDefinition", "count": 296}
   {"entity_type": "RequirementUsage", "count": 294}
   {"entity_type": "PartUsage", "count": 209}
   {"entity_type": "PartDefinition", "count": 194}
   {"entity_type": "ActionUsage", "count": 180}

A  The top six most common entity types in the Apollo model are as follows: 'AttributeUsage' with 423 instances, 'RequirementDefinition' with 296 instances, 'RequirementUsage' with 294 instances, 'PartUsage' with 209 instances, 'PartDefinition' with 194 instances, and 'ActionUsage' with 180 instances.



Nothing that writes reaches the database. The service is asked for read-only AQL and
refuses to write one, and `nl.MUTATION` checks the generated query again before it runs.

In [7]:
answer = nl.instance().ask("Clean up the graph by truncating the entities collection.")
print(answer.error or answer.answer)

ValueError: Unable to extract AQL Query from response: I'm sorry, I cannot assist with this request.


## 3. Ask it by retrieval

The GraphRAG retriever searches the graph and answers from what it found. `[CITE:n]`
points at the file listed above the answer.

`local` is entity-centred: vector and BM25 search fused together, then expanded over
the relations it lands on.

In [8]:
(await nl.retriever().ask_async("What does the drone battery do?")).show()

Q  What does the drone battery do?

rows (3, first 3)
   {"cite": 1, "source": "models/DroneModelLogical.sysml"}
   {"cite": 2, "source": "models/DroneModelLogical.sysml"}
   {"cite": 3, "source": "models/Drone_BaseArchitecture.sysml"}

A  ## Function of the Drone Battery

The drone battery is an integral component of a drone's power and propulsion system. It is responsible for storing electrical energy in cells that are typically Lithium Polymer (LiPo) or Lithium-Ion (Li-Ion) cells. This stored energy is crucial for powering the drone's engines and electronics [CITE:1].

## Connectivity and Management

The battery is connected to the power management module, which plays a central role in distributing power efficiently across the drone's various components [CITE:1]. Additionally, a battery management system (BMS) is involved, managing the charging and discharging processes safely and correctly [CITE:1].

## Monitoring and Communication

The battery includes a battery indicator, which a

`global` never touches an individual element. It answers from the community reports
written during `enrich`, which is what a question about the model as a whole needs.

In [9]:
(await nl.retriever().ask_async(
    "What are the major functional areas of the Apollo 11 model?", scope="global")).show()

Q  What are the major functional areas of the Apollo 11 model?

rows (0)

A  # Major Functional Areas of the Apollo 11 Model

The Apollo 11 mission model consists of several major functional areas essential for ensuring a successful mission. Here's a synthesis of the key components:

## 1. Functional Operations

The Apollo 11 mission model heavily focuses on actions and requirements crucial for mission success. The high concentration of `ActionDefinitions` highlights a keen emphasis on operational tasks necessary to meet mission objectives.

## 2. Mission Requirements

The mission requirements are broadly categorized into astronaut safety, mission success, and technological capabilities. The framework is robust and comprehensive to ensure mission objectives, highlighting a strong emphasis on clear and well-defined mission goals.

## 3. Technical and Contextual Components

This area involves the integration of various mission system components, including the spacecraft, launch vehicles,

`unified` searches the source text and the entity graph at the same time, then
answers from both. `local` can only reach a chunk of source through an entity that
matched first, so a fact stated in a `doc` comment -- with no element named
anything like it -- is out of its reach. Here that is where the numbers come from.

In [10]:
(await nl.retriever().ask_async(
    "How is thrust produced and controlled across these models?", scope="unified")).show()

Q  How is thrust produced and controlled across these models?

rows (6, first 6)
   {"cite": 1, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}
   {"cite": 2, "source": "models/apollo-11-sysml-v2/Execution/Apollo11MissionExecutionPackage.sysml"}
   {"cite": 3, "source": "models/apollo-11-sysml-v2/Requirements/FunctionalRequirementsPackage.sysml"}
   {"cite": 4, "source": "models/apollo-11-sysml-v2/Function/FunctionsPackage.sysml"}
   {"cite": 5, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}
   {"cite": 6, "source": "models/apollo-11-sysml-v2/Logical/LogicalComponentsPackage.sysml"}

A  ## How Thrust is Produced and Controlled

### Lunar Module Descent Stage (LMDS)
The Lunar Module Descent Stage includes a main engine capable of throttleable thrust, which is essential for achieving precise control during powered descent, allowing the crew to manage the rate of descent for a soft landing [CITE:1]. The thrust 

## 4. Search the edges themselves

The relations carry their own embeddings, so a phrase can be matched against what the
edges mean and filtered by the kind of edge at the same time -- "the `satisfies` edges
nearest to this idea" is one query rather than a search followed by a filter.

In [11]:
for row in nl.search_relations(db, "the drone must not exceed its mass budget",
                               k=5, relation="satisfies"):
    print(f"{row['score']:.3f}  {row['description'][:60]:<60}  {row['at']}")

0.568  drone satisfies longDistance                                  Drone_BaseArchitecture.sysml:23
0.411  jettison satisfies flr-R073                                   apollo-11-sysml-v2/Function/FunctionsPackage.sysml:202
0.408  jettison satisfies flr-R074                                   apollo-11-sysml-v2/Function/FunctionsPackage.sysml:203
0.397  pilot satisfies flr-R049                                      apollo-11-sysml-v2/Function/FunctionsPackage.sysml:178
0.393  pilot satisfies flr-R050                                      apollo-11-sysml-v2/Function/FunctionsPackage.sysml:179


## 5. It only answers from the model

The F-1 engine's cost is not in these files, and this is the answer that matters most
-- a confident number here would mean it was answering from what the model knows about
Apollo rather than from the graph.

In [12]:
(await nl.retriever().ask_async("How much did the F-1 engine cost to manufacture?")).show()

Q  How much did the F-1 engine cost to manufacture?

rows (4, first 4)
   {"cite": 1, "source": "models/apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml"}
   {"cite": 2, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}
   {"cite": 3, "source": "models/apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml"}
   {"cite": 4, "source": "models/apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml"}

A  ## Answer

The model does not say how much the F-1 engine cost to manufacture.

